> **교육 목표** Walk-forward로 방법을 같은 조건에서 비교하고, 복잡한 모델이 단순한 naive 기준선을 정말 이기는지 봅니다.
>
> **핵심 내용** 첫 주 손으로 예측 → 네 방법 Walk-forward → MAE·MAPE → 계절 naive 대비 개선율 → MAPE가 큰 이유

In [1]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D5_05 Walk-forward 평가와 naive 기준선

> **오늘 질문:** 매주 다음 7일을 예측한다면, 어떤 방법이 가장 잘 맞을까? 단순한 방법보다 정말 나을까?

**평가 방식 (모든 방법 동일)**
- Test: 시간 순 마지막 20% (10월 20일 ~ 12월 31일, 73일)
- 매주 '그때까지의 데이터'로 다음 7일을 예측 → 실제와 비교 → 한 주 이동

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# 1교시에서 한 시간 정리 (00:00 기록 바로잡기 → 구간 시작 시각으로)
se = pd.read_csv("../../data/steel_energy.csv")
se["date"] = pd.to_datetime(se["date"], format="%d/%m/%Y %H:%M")
midnight = (se["date"].dt.hour == 0) & (se["date"].dt.minute == 0)
se.loc[midnight, "date"] += pd.Timedelta(days=1)
se["date"] -= pd.Timedelta(minutes=15)
se = se.set_index("date").sort_index()
daily = se["usage_kwh"].resample("D").sum()

cut = int(len(daily) * 0.8)      # Test 시작 위치

## 1. 첫 주만 손으로 예측해 보기 ✍️

Test 첫 주(7일)를 두 가지 기준선으로 예측합니다.
- **naive**: 7일 모두 '마지막 날 값'
- **계절 naive**: 7일 각각 '지난주 같은 요일 값'

> 변수 이름: `history`, `actual`

In [ ]:
# 힌트: 시리즈.iloc[-1] 은 마지막 값, 시리즈.iloc[-7:] 은 마지막 7개

## 2. Walk-forward로 네 방법 예측 📋

Test 기간을 7일씩 이동하며 네 방법을 모두 예측합니다. ARIMA·SARIMA는 매주 그때까지의 데이터로 다시 학습합니다. (10초 정도)

In [ ]:
rows = []
for start in range(cut, len(daily), 7):
    history = daily.iloc[:start]                      # 예측 시점까지의 데이터만
    days = daily.index[start:start + 7]               # 예측할 날짜 (마지막 주는 7일보다 짧을 수 있음)
    n = len(days)

    arima = ARIMA(history, order=(1, 1, 1)).fit()
    sarima = SARIMAX(history, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7)).fit(disp=False)

    week = pd.DataFrame({
        "naive": history.iloc[-1],
        "계절 naive": history.iloc[-7:].values[:n],
        "ARIMA": arima.forecast(n).values,
        "SARIMA": sarima.forecast(n).values,
    }, index=days)
    rows.append(week)

preds = pd.concat(rows)
print(f"예측한 날: {len(preds)}일")
preds.head(8).round(0)

## 3. MAE와 MAPE ✍️

- MAE = |실제 − 예측|의 평균 (kWh)
- MAPE = |실제 − 예측| ÷ 실제의 평균 (%)

`sub(..., axis=0)`은 각 열에서 같은 날의 실제 값을 뺍니다.

> 변수 이름: `actual`, `error`, `score`

In [ ]:
# 힌트: 표.sub(시리즈, axis=0), 표.div(시리즈, axis=0)

## 4. 계절 naive 대비 개선율 ✍️

보고서에는 MAE만이 아니라 '기준선 대비 몇 % 나아졌나'를 함께 씁니다. 음수면 기준선보다 나쁜 것입니다.

In [ ]:
# 힌트: (1 - MAE / 기준선 MAE) * 100

## 5. 예측-실제 그래프 ✍️

## 6. MAPE가 큰 이유 📋

사용량이 아주 적은 날의 % 오차를 봅니다.

In [ ]:
ape = error["SARIMA"].div(actual) * 100
pd.DataFrame({"실제": actual, "SARIMA 예측": preds["SARIMA"], "% 오차": ape}).sort_values("실제").head(6).round(0)

## 정리 💬

**Q1.** 네 방법의 순위는? ARIMA는 계절 naive를 이겼나요? (슬라이드의 "naive가 이긴다?")

> 답:


**Q2.** MAPE가 60~160%로 매우 큰 이유는? 이 데이터에서는 어떤 지표로 보고해야 할까요?

> 답:
